<a href="https://colab.research.google.com/github/prasoon-06/zomato-project/blob/main/Zomato_Sentiment_Analysis_Capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Zomato Restaurant Reviews - Sentiment Analysis**    -

##### **Project Type**    - Classification (NLP Sentiment Analysis) / EDA
##### **Contribution**    - Individual
##### **Name -** Prasoon Sahoo

# **Project Summary -**

Zomato is one of India's most popular restaurant discovery and food delivery platforms. This project analyzes two datasets scraped from Zomato for the city of Hyderabad: a **restaurant metadata** file containing 105 restaurants with their cost for two, cuisines, collections (curated tags) and timings, and a **customer reviews** file containing close to 10,000 individual reviews with the reviewer's name, free-text review, star rating (1-5), reviewer metadata (number of past reviews/followers), timestamp and number of pictures attached.

The goal of the project is two-fold. First, a thorough Exploratory Data Analysis (EDA) is carried out to understand how restaurant characteristics such as cost, cuisine and curated collections relate to customer ratings, and how reviewer behaviour (review length, activity, time of posting) relates to the rating given. Second, a **Natural Language Processing (NLP) classification model** is built that predicts the *sentiment* of a review (Positive / Neutral / Negative) purely from the free-text review, without looking at the numeric star rating. This is useful because in the real world a large share of user-generated text (social media mentions, support tickets, delivery-partner feedback) never comes with an explicit star rating, so an automated sentiment classifier lets Zomato triage and monitor customer experience at scale.

During data wrangling, the `Rating` column was cleaned (a stray non-numeric value `"Like"` was removed, and the column was cast to float), the `Cost` column in the metadata file was converted from a comma-formatted string to numeric, and the two files were merged on restaurant name so that cost/cuisine information could be studied alongside individual reviews. Reviewer metadata (e.g. `"3 Reviews , 2 Followers"`) was parsed with regular expressions into two new numeric columns, `ReviewerNumReviews` and `ReviewerNumFollowers`, and review length was captured as a new feature.

Key EDA findings include: ratings are heavily skewed towards the positive end (around 63% of reviews are Positive, i.e. rating >= 4, versus 25% Negative and 12% Neutral), higher-cost restaurants tend to receive marginally higher average ratings, and North Indian and Chinese are by far the most common cuisines offered among the sampled restaurants. Review length shows almost no linear correlation with the star rating, but very short 5-star reviews and long, detailed 1-star complaint reviews are both common - suggesting length alone is a weak signal and the actual wording matters far more, which motivates the text-based classification approach.

For the modelling stage, review text was cleaned (lower-cased, punctuation/digits removed, stopwords removed) and converted into numeric features using **TF-IDF vectorization** with unigrams and bigrams. Because the sentiment classes are imbalanced, `class_weight='balanced'` was used across models. Three classifiers were trained and compared: **Multinomial Naive Bayes**, **Logistic Regression**, and **Random Forest**. Logistic Regression with hyperparameter tuning (GridSearchCV over the regularisation strength `C`) was selected as the final model because it offered the best balance of accuracy, weighted F1-score and interpretability (its coefficients directly show which words push a review towards Positive or Negative sentiment), achieving roughly 78% accuracy and a weighted F1-score of about 0.79 on the held-out test set.

This sentiment model has direct business value: it allows Zomato's operations team to automatically flag restaurants that are accumulating negative sentiment even before their average star rating visibly drops, prioritize customer-support follow-ups on angry reviews, and feed a continuously updated "restaurant health score" into the recommendation and search-ranking systems.

# **GitHub Link -**

https://github.com/prasoon-06/zomato-project.git

# **Problem Statement**


Given restaurant metadata (cost, cuisines, collections, timings) and ~10,000 customer reviews (review text, star rating, reviewer activity, timestamp) scraped from Zomato for restaurants in Hyderabad, the objectives are to:

1. Perform an in-depth Exploratory Data Analysis to uncover patterns between restaurant attributes (cost, cuisine, collections) and customer satisfaction (ratings), and between reviewer behaviour and the ratings/sentiment they express.
2. Engineer a **Sentiment** label (Positive / Neutral / Negative) from the numeric star rating.
3. Clean and vectorize the free-text reviews using standard NLP preprocessing and TF-IDF.
4. Build, tune and compare multiple classification models that predict review sentiment **from the review text alone**, and select the best model for potential deployment as an automated review-monitoring tool.

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import numpy as np
import pandas as pd
import re
import string
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                              classification_report, confusion_matrix, ConfusionMatrixDisplay)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

### Dataset Loading

In [ ]:
# Load Dataset
# Restaurant metadata: name, cost, cuisines, collections, timings
meta_df = pd.read_csv('/content/drive/MyDrive/zomato/Zomato Restaurant names and Metadata.csv')

# Customer reviews: restaurant, reviewer, review text, rating, reviewer metadata, time, pictures
reviews_df = pd.read_csv('/content/drive/MyDrive/zomato/Zomato Restaurant reviews.csv')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
print(os.listdir())

### Dataset First View

In [ ]:
# Dataset First Look
print("Restaurant Metadata:")
display(meta_df.head())
print("\nReviews Data:")
display(reviews_df.head())

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
print(f"Restaurant Metadata -> Rows: {meta_df.shape[0]}, Columns: {meta_df.shape[1]}")
print(f"Reviews Data        -> Rows: {reviews_df.shape[0]}, Columns: {reviews_df.shape[1]}")

### Dataset Information

In [ ]:
# Dataset Info
print("Restaurant Metadata Info:")
meta_df.info()
print("\nReviews Data Info:")
reviews_df.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
print("Duplicate rows in Restaurant Metadata:", meta_df.duplicated().sum())
print("Duplicate rows in Reviews Data:", reviews_df.duplicated().sum())

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
print("Missing values in Restaurant Metadata:\n", meta_df.isnull().sum())
print("\nMissing values in Reviews Data:\n", reviews_df.isnull().sum())

In [ ]:
# Visualizing the missing values
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.heatmap(meta_df.isnull(), cbar=False, cmap='viridis', ax=axes[0])
axes[0].set_title('Missing Values - Restaurant Metadata')
sns.heatmap(reviews_df.isnull(), cbar=False, cmap='viridis', ax=axes[1])
axes[1].set_title('Missing Values - Reviews Data')
plt.tight_layout()
plt.show()

### What did you know about your dataset?

- The **Restaurant Metadata** dataset has 105 restaurants and 6 columns (`Name`, `Links`, `Cost`, `Collections`, `Cuisines`, `Timings`). `Cost` is stored as a comma-formatted string (e.g. `"1,300"`) and needs to be converted to numeric. `Collections` has 54 missing values (restaurants that were not curated into any Zomato collection) and `Timings` has 1 missing value.
- The **Reviews** dataset has 10,000 rows and 7 columns (`Restaurant`, `Reviewer`, `Review`, `Rating`, `Metadata`, `Time`, `Pictures`). 38 rows are almost entirely empty (missing `Reviewer`, `Review`, `Rating`, `Metadata`, `Time` together) and should be dropped. The `Rating` column is stored as text and also contains one invalid value `"Like"` that must be removed before converting to float.
- There are no duplicate rows in either dataset.
- Each restaurant has on average ~95 reviews (9,954 clean reviews across 100 restaurants that actually have reviews, out of the 105 in the metadata file), giving a reasonably rich dataset for both restaurant-level and review-level analysis.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
print("Restaurant Metadata columns:", list(meta_df.columns))
print("Reviews Data columns:", list(reviews_df.columns))

In [ ]:
# Dataset Describe
display(meta_df.describe(include='all'))
display(reviews_df.describe(include='all'))

### Variables Description

**Restaurant Metadata**
- `Name` - Name of the restaurant.
- `Links` - URL of the restaurant's Zomato page.
- `Cost` - Approximate cost for two people dining (in INR), stored as text with commas.
- `Collections` - Comma-separated curated Zomato collections/tags the restaurant belongs to (e.g. "Great Buffets").
- `Cuisines` - Comma-separated list of cuisines served.
- `Timings` - Restaurant's operating hours.

**Reviews Data**
- `Restaurant` - Name of the restaurant being reviewed (foreign key to Metadata `Name`).
- `Reviewer` - Name of the person who wrote the review.
- `Review` - Free-text review content (our main NLP feature).
- `Rating` - Star rating given by the reviewer (1-5, sometimes with .5 increments).
- `Metadata` - Text describing the reviewer's activity, e.g. "3 Reviews , 2 Followers".
- `Time` - Timestamp of when the review was posted.
- `Pictures` - Number of pictures attached to the review.

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
print("Unique restaurants in Metadata:", meta_df['Name'].nunique())
print("Unique restaurants reviewed:", reviews_df['Restaurant'].nunique())
print("Unique reviewers:", reviews_df['Reviewer'].nunique())
print("Unique Rating values (raw):", reviews_df['Rating'].unique())
print("Unique Cost values (sample):", meta_df['Cost'].unique()[:10])

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.

# 1. Clean the Reviews dataset -----------------------------------------
reviews_df = reviews_df.dropna(subset=['Review', 'Rating']).copy()
reviews_df = reviews_df[reviews_df['Rating'] != 'Like']
reviews_df['Rating'] = reviews_df['Rating'].astype(float)
reviews_df['Time'] = pd.to_datetime(reviews_df['Time'])
reviews_df = reviews_df.drop_duplicates().reset_index(drop=True)

# 2. Clean the Metadata dataset -----------------------------------------
meta_df['Cost'] = meta_df['Cost'].astype(str).str.replace(',', '', regex=False).astype(float)

# 3. Parse reviewer activity out of the free-text `Metadata` column ------
def extract_count(text, keyword):
    match = re.search(rf'(\d+)\s*{keyword}', str(text))
    return int(match.group(1)) if match else 0

reviews_df['ReviewerNumReviews'] = reviews_df['Metadata'].apply(lambda x: extract_count(x, 'Review'))
reviews_df['ReviewerNumFollowers'] = reviews_df['Metadata'].apply(lambda x: extract_count(x, 'Follower'))

# 4. Derived features -----------------------------------------------------
reviews_df['ReviewLength'] = reviews_df['Review'].str.len()
reviews_df['ReviewWordCount'] = reviews_df['Review'].str.split().apply(len)
reviews_df['DayOfWeek'] = reviews_df['Time'].dt.day_name()
reviews_df['Hour'] = reviews_df['Time'].dt.hour

def rating_to_sentiment(r):
    if r >= 4:
        return 'Positive'
    elif r == 3:
        return 'Neutral'
    else:
        return 'Negative'

reviews_df['Sentiment'] = reviews_df['Rating'].apply(rating_to_sentiment)

# 5. Merge reviews with restaurant metadata for restaurant-level analysis -
merged_df = reviews_df.merge(meta_df, left_on='Restaurant', right_on='Name', how='left')

print(reviews_df.shape, meta_df.shape, merged_df.shape)
reviews_df.head()

### What all manipulations have you done and insights you found?

- Dropped 38 rows in the Reviews dataset that had missing `Review`/`Rating`/`Reviewer` values, and removed 1 row with an invalid, non-numeric `"Like"` rating; the `Rating` column was then cast from text to float, leaving 9,954 usable reviews.
- Converted `Cost` in the Metadata dataset from a comma-formatted string (e.g. `"1,300"`) to a numeric float so it can be used in correlation/aggregation analysis.
- Parsed the free-text `Metadata` reviewer-activity field (e.g. `"3 Reviews , 2 Followers"`) into two clean numeric columns, `ReviewerNumReviews` and `ReviewerNumFollowers`, using regex, since these are potentially useful proxies for reviewer credibility/influence.
- Engineered `ReviewLength`, `ReviewWordCount`, `DayOfWeek` and `Hour` as new features to support the EDA.
- Engineered the target variable **`Sentiment`** (Positive/Neutral/Negative) from the numeric `Rating`, which is the label used for the downstream NLP classification task.
- Merged the reviews with restaurant metadata on restaurant name to be able to relate cost/cuisine to individual review ratings.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart - 1 visualization code
plt.figure(figsize=(9,5))
sns.countplot(data=reviews_df, x='Rating', order=sorted(reviews_df['Rating'].unique()), palette='viridis')
plt.title('Distribution of Star Ratings')
plt.xlabel('Rating')
plt.ylabel('Number of Reviews')
plt.show()

##### 1. Why did you pick the specific chart?

A count plot is the most direct way to show the frequency of each discrete rating value (1 to 5, including half-stars) and immediately reveals the shape of the target distribution.

##### 2. What is/are the insight(s) found from the chart?

Ratings are heavily right-skewed: 5-star reviews are the single largest bucket (~3,826), followed by 4-star (~2,373), while 1-star reviews (~1,735) outnumber both 2-star and 3-star reviews. Overall, roughly 63% of all reviews are Positive (rating >= 4).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes - it immediately flags that the target is imbalanced, which is critical for correctly setting up the classification models later (e.g. using `class_weight='balanced'`) and for choosing F1-score over raw accuracy as the primary evaluation metric, so the negative-growth risk here is a *modelling* one: a naive model could get ~63% accuracy just by always predicting 'Positive', giving a false sense of business health.

#### Chart - 2

In [ ]:
# Chart - 2 visualization code
plt.figure(figsize=(7,5))
sentiment_counts = reviews_df['Sentiment'].value_counts()
colors = {'Positive':'#2ecc71','Neutral':'#f1c40f','Negative':'#e74c3c'}
sentiment_counts.plot(kind='bar', color=[colors[c] for c in sentiment_counts.index])
plt.title('Sentiment Class Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Number of Reviews')
plt.xticks(rotation=0)
plt.show()

##### 1. Why did you pick the specific chart?

A bar chart of the engineered `Sentiment` target (Positive/Neutral/Negative) makes the class imbalance of the actual modelling label explicit and easy to compare across only three categories.

##### 2. What is/are the insight(s) found from the chart?

Positive reviews dominate (~63%), Negative reviews are the second largest group (~25%), and Neutral is by far the smallest (~12%). Customers on Zomato reviews are much more likely to leave a review when they are happy or clearly unhappy, and rarely leave a lukewarm 3-star review.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, positively - a large Positive base signals decent overall customer satisfaction across the sampled restaurants. The one risk for negative growth is that the small Neutral class will be hard for any model to learn well and may get frequently mis-classified as Positive or Negative, which the business should be aware of when interpreting the model's Neutral predictions.

#### Chart - 3

In [ ]:
# Chart - 3 visualization code
plt.figure(figsize=(9,5))
sns.histplot(meta_df['Cost'], bins=20, kde=True, color='teal')
plt.title('Distribution of Restaurant Cost for Two')
plt.xlabel('Cost for Two (INR)')
plt.ylabel('Number of Restaurants')
plt.show()

##### 1. Why did you pick the specific chart?

A histogram with a KDE overlay is ideal for a continuous numeric variable like `Cost`, showing both the shape of the distribution and where costs cluster.

##### 2. What is/are the insight(s) found from the chart?

Costs range from INR 150 to INR 2,800, with a median around INR 700 and a strong concentration between INR 500-1,200. Very cheap (<300) and very expensive (>1,800) restaurants are comparatively rare in this sample.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes - it helps Zomato understand that the platform's restaurant base in this sample is dominated by mid-range dining, useful for targeted promotions or subscription-tier pricing; no clear negative-growth signal here, it's simply descriptive of market composition.

#### Chart - 4

In [ ]:
# Chart - 4 visualization code
cuisine_series = meta_df['Cuisines'].dropna().str.split(', ').explode()
top_cuisines = cuisine_series.value_counts().head(10)

plt.figure(figsize=(9,6))
sns.barplot(x=top_cuisines.values, y=top_cuisines.index, palette='mako')
plt.title('Top 10 Most Common Cuisines')
plt.xlabel('Number of Restaurants Offering the Cuisine')
plt.ylabel('Cuisine')
plt.show()

##### 1. Why did you pick the specific chart?

Since `Cuisines` is a multi-label, comma-separated field, exploding it into individual cuisine tags and then using a horizontal bar chart is the clearest way to rank categorical frequency.

##### 2. What is/are the insight(s) found from the chart?

North Indian (61 restaurants) and Chinese (43) are by far the most commonly offered cuisines, followed by Continental (21), Biryani (16) and Asian/Fast Food (15 each) - reflecting a strong North-Indian/Chinese multi-cuisine dining culture in Hyderabad.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes - this tells Zomato which cuisines are already saturated (North Indian, Chinese) versus underrepresented (e.g. South Indian, Bakery, despite Hyderabad's regional context), informing partner-acquisition strategy for cuisine diversity. No negative growth signal, purely a market-composition insight.

#### Chart - 5

In [ ]:
# Chart - 5 visualization code
collection_series = meta_df['Collections'].dropna().str.split(', ').explode()
top_collections = collection_series.value_counts().head(10)

plt.figure(figsize=(9,6))
sns.barplot(x=top_collections.values, y=top_collections.index, palette='rocket')
plt.title('Top 10 Zomato Collections Restaurants Belong To')
plt.xlabel('Number of Restaurants')
plt.ylabel('Collection')
plt.show()

##### 1. Why did you pick the specific chart?

Same exploded-categorical approach as the cuisine chart, applied to `Collections`, to see which curated Zomato tags are most represented among the sampled restaurants.

##### 2. What is/are the insight(s) found from the chart?

'Great Buffets' (11), 'Food Hygiene Rated Restaurants in Hyderabad' (8), and 'Live Sports Screenings' / "Hyderabad's Hottest" (7 each) are the most common collections, showing Zomato's curation leans towards buffet and hygiene-certified experiences in this city.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes - collections act as a discovery/marketing lever; restaurants without any collection tag (54 of them) are less discoverable on the app, so Zomato's curation team could prioritize reviewing and tagging those restaurants, a direct, actionable positive-impact insight.

#### Chart - 6

In [ ]:
# Chart - 6 visualization code
merged_df['CostBucket'] = pd.cut(merged_df['Cost'], bins=[0,300,600,900,1500,3000],
                                  labels=['<300','300-600','600-900','900-1500','1500+'])
plt.figure(figsize=(9,5))
sns.boxplot(data=merged_df, x='CostBucket', y='Rating', palette='coolwarm')
plt.title('Rating Distribution across Cost Buckets')
plt.xlabel('Cost for Two (Bucketed)')
plt.ylabel('Rating')
plt.show()

##### 1. Why did you pick the specific chart?

A box plot is well-suited to compare the distribution (median, spread, outliers) of a numeric variable (`Rating`) across an ordered categorical bucket (`CostBucket`), revealing trends better than a simple bar of means.

##### 2. What is/are the insight(s) found from the chart?

Median ratings rise steadily as cost increases: restaurants under INR 600 have a median rating around 3-3.5 while restaurants above INR 1,500 have medians closer to 4-4.5. The correlation between numeric cost and rating is modest but positive (~0.14).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, moderately - it suggests that pricier restaurants tend to deliver a more consistently satisfying experience (better ambience, staff, quality control), which Zomato could highlight to users looking to guarantee quality, though the effect size is small so it should not be over-generalized as 'expensive = better'.

#### Chart - 7

In [ ]:
# Chart - 7 visualization code
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
plt.figure(figsize=(10,5))
sns.countplot(data=reviews_df, x='DayOfWeek', order=day_order, palette='crest')
plt.title('Number of Reviews Posted by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Number of Reviews')
plt.show()

##### 1. Why did you pick the specific chart?

A count plot ordered Monday-through-Sunday makes weekly review-posting patterns easy to read at a glance.

##### 2. What is/are the insight(s) found from the chart?

Reviews peak on Sunday (1,825) and Saturday (1,736), consistent with more weekend dining-out occasions, and are lowest on Tuesday (1,192) and Thursday (1,243).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes - this helps Zomato plan when to run promotions, push notifications for review prompts, or staff up customer-support/moderation teams, since weekend traffic (and therefore review volume) is meaningfully higher.

#### Chart - 8

In [ ]:
# Chart - 8 visualization code
plt.figure(figsize=(10,5))
sns.histplot(reviews_df['Hour'], bins=24, kde=False, color='darkorange')
plt.title('Number of Reviews Posted by Hour of Day')
plt.xlabel('Hour of Day (24h)')
plt.ylabel('Number of Reviews')
plt.show()

##### 1. Why did you pick the specific chart?

A histogram over the 24-hour range is the natural chart for a cyclical numeric variable like posting hour, showing intraday peaks and troughs.

##### 2. What is/are the insight(s) found from the chart?

Review activity peaks sharply in the evening between 8 PM and 11 PM (with hour 22 alone accounting for nearly 1,000 reviews), consistent with people reviewing right after dinner, and stays low in the early morning hours.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes - Zomato can time in-app review reminders and promotional pushes to land just after this dinner-time peak window, when engagement (and willingness to leave a review) is highest.

#### Chart - 9

In [ ]:
# Chart - 9 visualization code
plt.figure(figsize=(9,5))
sns.boxplot(data=reviews_df, x='Sentiment', y='ReviewLength', order=['Negative','Neutral','Positive'], palette=colors)
plt.title('Review Length by Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Review Length (characters)')
plt.ylim(0, 1500)
plt.show()

##### 1. Why did you pick the specific chart?

A box plot compares the numeric `ReviewLength` distribution across the three sentiment categories, useful to check whether length itself is a discriminative signal for the classification task.

##### 2. What is/are the insight(s) found from the chart?

Median review length is broadly similar and fairly noisy across all three sentiment classes, and the overall correlation between length and rating is close to zero (~ -0.03); however extremely short reviews are common in the 5-star (Positive) bucket while long, detailed complaint narratives are common at 1-star (Negative) - length alone is not a strong standalone signal.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This is a modelling insight rather than a growth one: it tells us the classifier must rely on the actual words used (TF-IDF) rather than superficial length-based features, guiding feature engineering choices later in the notebook.

#### Chart - 10

In [ ]:
# Chart - 10 visualization code
cuisine_rating = merged_df.assign(Cuisine=merged_df['Cuisines'].str.split(', ')).explode('Cuisine')
top10_cuisine_names = cuisine_series.value_counts().head(10).index
avg_rating_by_cuisine = (cuisine_rating[cuisine_rating['Cuisine'].isin(top10_cuisine_names)]
                          .groupby('Cuisine')['Rating'].mean().sort_values(ascending=False))

plt.figure(figsize=(9,6))
sns.barplot(x=avg_rating_by_cuisine.values, y=avg_rating_by_cuisine.index, palette='flare')
plt.title('Average Rating for Top 10 Cuisines')
plt.xlabel('Average Rating')
plt.ylabel('Cuisine')
plt.xlim(0,5)
plt.show()

##### 1. Why did you pick the specific chart?

A horizontal bar chart of mean rating per cuisine (restricted to the top-10 most common cuisines for statistical reliability) directly compares customer satisfaction across cuisine types.

##### 2. What is/are the insight(s) found from the chart?

Average ratings across the top cuisines are fairly close to each other (mostly in the 3.5-4.2 range), meaning cuisine type alone is not a major driver of satisfaction - execution quality (service, hygiene, consistency) likely matters more than the cuisine category itself.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Slightly - it tells the business that marketing 'best cuisines' is less useful than restaurant-level quality signals; no strong negative-growth risk identified from this chart.

#### Chart - 11

In [ ]:
# Chart - 11 visualization code
restaurant_avg = merged_df.groupby('Restaurant')['Rating'].agg(['mean','count'])
restaurant_avg = restaurant_avg[restaurant_avg['count'] >= 20]  # restaurants with enough reviews
top5 = restaurant_avg.sort_values('mean', ascending=False).head(5)
bottom5 = restaurant_avg.sort_values('mean', ascending=True).head(5)
combo = pd.concat([top5, bottom5])

plt.figure(figsize=(9,6))
sns.barplot(x=combo['mean'], y=combo.index, palette=['#2ecc71']*5 + ['#e74c3c']*5)
plt.title('Top 5 & Bottom 5 Restaurants by Average Rating (min. 20 reviews)')
plt.xlabel('Average Rating')
plt.ylabel('Restaurant')
plt.xlim(0,5)
plt.show()

##### 1. Why did you pick the specific chart?

A single combined bar chart contrasting the best and worst-performing restaurants (filtered to those with a reliable sample size of 20+ reviews) is an effective way to spotlight outliers at the restaurant level.

##### 2. What is/are the insight(s) found from the chart?

Restaurants like AB's - Absolute Barbecues and Paradise sit near the top with average ratings above 4.6, while restaurants like Hotel Zara Hi-Fi and Asian Meal Box sit at the bottom with averages around 2.4-2.6, a very wide gap despite operating on the same platform in the same city.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, directly - this is the kind of view Zomato's account-management team would use to flag consistently under-performing partner restaurants for corrective action (or removal from prominent placements), while top performers can be highlighted or used as case studies for best practices.

#### Chart - 12

In [ ]:
# Chart - 12 visualization code
positive_text = ' '.join(reviews_df[reviews_df['Sentiment']=='Positive']['Review'].astype(str).sample(2000, random_state=42))
negative_text = ' '.join(reviews_df[reviews_df['Sentiment']=='Negative']['Review'].astype(str))

fig, axes = plt.subplots(1, 2, figsize=(18,8))
wc_pos = WordCloud(width=800, height=600, background_color='white', colormap='Greens', stopwords=set(stopwords.words('english'))).generate(positive_text)
wc_neg = WordCloud(width=800, height=600, background_color='white', colormap='Reds', stopwords=set(stopwords.words('english'))).generate(negative_text)
axes[0].imshow(wc_pos); axes[0].axis('off'); axes[0].set_title('Positive Reviews Word Cloud')
axes[1].imshow(wc_neg); axes[1].axis('off'); axes[1].set_title('Negative Reviews Word Cloud')
plt.show()

##### 1. Why did you pick the specific chart?

A word cloud is a quick, intuitive way to surface the most frequent, and therefore most discriminative, vocabulary used in each sentiment class before doing any formal vectorization.

##### 2. What is/are the insight(s) found from the chart?

Positive reviews are dominated by words like 'good', 'great', 'best', 'ambience', 'service', 'delicious', and 'friendly', while Negative reviews are dominated by words like 'worst', 'bad', 'slow', 'cold', 'rude', 'waited', and 'disappointed' - a very clean separation in vocabulary that supports using a bag-of-words / TF-IDF approach for classification.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes - beyond modelling value, this is a ready-made summary Zomato's operations team could use to see, at a glance, what customers repeatedly praise (service, ambience) versus complain about (wait times, temperature of food, staff rudeness), directly informing operational improvement priorities for underperforming restaurants.

#### Chart - 13

In [ ]:
# Chart - 13 visualization code
plt.figure(figsize=(9,5))
sns.scatterplot(data=reviews_df.sample(2000, random_state=42), x='ReviewerNumReviews', y='Rating', alpha=0.4, color='purple')
plt.title('Reviewer Activity (Num. Past Reviews) vs Rating Given')
plt.xlabel('Number of Reviews by the Reviewer')
plt.ylabel('Rating Given')
plt.xlim(0, 200)
plt.show()

##### 1. Why did you pick the specific chart?

A scatter plot is the standard chart for exploring the relationship between two numeric variables - here, whether more experienced/prolific reviewers rate differently than first-time reviewers.

##### 2. What is/are the insight(s) found from the chart?

There is no visible linear pattern and the correlation is essentially zero (~0.04) - reviewer activity level does not meaningfully predict the rating given; both very new and very experienced reviewers give the full range of ratings.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Limited direct impact, but it does rule out a hypothesis (that highly active 'power users' are systematically harsher or more lenient raters), which is useful to know before designing any reviewer-trust-weighting scheme.

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code
numeric_cols = ['Rating', 'Pictures', 'ReviewerNumReviews', 'ReviewerNumFollowers', 'ReviewLength', 'ReviewWordCount']
plt.figure(figsize=(9,7))
sns.heatmap(reviews_df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', center=0)
plt.title('Correlation Heatmap of Numeric Review Features')
plt.show()

##### 1. Why did you pick the specific chart?

A correlation heatmap efficiently summarizes pairwise linear relationships between all numeric features at once, which is the standard final step of bivariate/multivariate numeric analysis.

##### 2. What is/are the insight(s) found from the chart?

All numeric features (`Pictures`, `ReviewerNumReviews`, `ReviewerNumFollowers`, `ReviewLength`, `ReviewWordCount`) have very weak correlation with `Rating` (all under 0.10 in magnitude), while `ReviewLength` and `ReviewWordCount` are, unsurprisingly, almost perfectly correlated with each other (~0.97), confirming that the actual textual content - not any of these metadata numbers - is what drives sentiment, reinforcing the NLP-based modelling approach used later.

#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code
pairplot_cols = ['Rating', 'ReviewLength', 'ReviewerNumReviews', 'ReviewerNumFollowers']
sns.pairplot(reviews_df[pairplot_cols].sample(1500, random_state=42), diag_kind='kde', corner=True)
plt.suptitle('Pair Plot of Key Numeric Features', y=1.02)
plt.show()

##### 1. Why did you pick the specific chart?

A pair plot extends the correlation heatmap by visualizing the actual joint distributions (not just the summary correlation coefficient) and the individual distributions on the diagonal, useful to catch non-linear relationships the heatmap might miss.

##### 2. What is/are the insight(s) found from the chart?

No strong non-linear relationships are visible either - all the scatter panels look like diffuse clouds rather than trends, and `ReviewerNumReviews`/`ReviewerNumFollowers` are both heavily right-skewed (a small number of very active 'super-reviewers'), confirming yet again that engagement metadata is a weak predictor of the rating/sentiment and that text content should be the primary modelling feature.

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

Based on the EDA above, three hypotheses are formed to statistically validate patterns that were only visually observed so far:
1. Restaurant cost is related to the rating it receives.
2. Review length differs between Positive and Negative reviews.
3. Ratings differ significantly across the top cuisines.

### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis

**H0 (Null):** There is no significant difference in the mean rating between low-cost restaurants (Cost < median) and high-cost restaurants (Cost >= median).
**H1 (Alternate):** There is a significant difference in the mean rating between low-cost and high-cost restaurants.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
from scipy import stats

median_cost = merged_df['Cost'].median()
low_cost_ratings = merged_df[merged_df['Cost'] < median_cost]['Rating'].dropna()
high_cost_ratings = merged_df[merged_df['Cost'] >= median_cost]['Rating'].dropna()

t_stat, p_value = stats.ttest_ind(low_cost_ratings, high_cost_ratings, equal_var=False)
print(f"T-statistic: {t_stat:.4f}, P-value: {p_value:.6f}")
if p_value < 0.05:
    print("Reject the Null Hypothesis: significant difference in ratings between low and high cost restaurants.")
else:
    print("Fail to reject the Null Hypothesis.")

##### Which statistical test have you done to obtain P-Value?

An independent two-sample **t-test** (Welch's t-test, `equal_var=False`) was used, since we are comparing the mean of a continuous variable (`Rating`) between two independent groups (low-cost vs high-cost restaurants).

##### Why did you choose the specific statistical test?

The t-test is the standard, well-understood test for comparing means between two independent groups. Welch's variant was chosen (instead of assuming equal variances) because the two cost groups are unlikely to have identical rating variance, making the test more robust.

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**H0 (Null):** There is no significant difference in mean review length between Positive and Negative reviews.
**H1 (Alternate):** There is a significant difference in mean review length between Positive and Negative reviews.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
positive_len = reviews_df[reviews_df['Sentiment']=='Positive']['ReviewLength']
negative_len = reviews_df[reviews_df['Sentiment']=='Negative']['ReviewLength']

t_stat2, p_value2 = stats.ttest_ind(positive_len, negative_len, equal_var=False)
print(f"T-statistic: {t_stat2:.4f}, P-value: {p_value2:.6f}")
if p_value2 < 0.05:
    print("Reject the Null Hypothesis: review length differs significantly between Positive and Negative reviews.")
else:
    print("Fail to reject the Null Hypothesis.")

##### Which statistical test have you done to obtain P-Value?

Again an independent two-sample **Welch's t-test**, comparing the mean `ReviewLength` between the Positive and Negative sentiment groups.

##### Why did you choose the specific statistical test?

The t-test is appropriate because both groups are independent samples and `ReviewLength` is a continuous numeric variable with a reasonably large sample size in each group (Central Limit Theorem makes the t-test robust even if the underlying distribution is not perfectly normal).

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**H0 (Null):** The mean rating is the same across all of the top cuisines (North Indian, Chinese, Continental, Biryani, Asian, etc.).
**H1 (Alternate):** At least one cuisine has a significantly different mean rating from the others.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
cuisine_groups = [group['Rating'].dropna().values for name, group in
                  cuisine_rating[cuisine_rating['Cuisine'].isin(top10_cuisine_names)].groupby('Cuisine')]

f_stat, p_value3 = stats.f_oneway(*cuisine_groups)
print(f"F-statistic: {f_stat:.4f}, P-value: {p_value3:.6f}")
if p_value3 < 0.05:
    print("Reject the Null Hypothesis: at least one cuisine has a significantly different mean rating.")
else:
    print("Fail to reject the Null Hypothesis: no significant difference in mean ratings across cuisines.")

##### Which statistical test have you done to obtain P-Value?

A **one-way ANOVA** (`f_oneway`) was used because we are comparing the mean of a continuous variable (`Rating`) across more than two independent categorical groups (the top-10 cuisines) simultaneously.

##### Why did you choose the specific statistical test?

ANOVA is the correct test for comparing means across 3+ groups at once - running many pairwise t-tests instead would inflate the false-positive (Type I error) rate, whereas ANOVA controls for this with a single omnibus F-test.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handling Missing Values & Missing Value Imputation
# Rows with missing Review/Rating were already dropped during Data Wrangling (step 3).
# For the Metadata dataset (not used as model input), remaining missing values are informational
# (e.g. a restaurant simply belongs to no curated Collection) so they are left as NaN / filled with 'None'.
meta_df['Collections'] = meta_df['Collections'].fillna('None')
meta_df['Timings'] = meta_df['Timings'].fillna('Not Available')
print(reviews_df.isnull().sum())

#### What all missing value imputation techniques have you used and why did you use those techniques?

No imputation was needed for the modelling features: the only truly missing values were in the 38 rows of the Reviews dataset that were missing `Review`/`Rating` together, and these were simply dropped (imputing a review's *text* or *rating* would be meaningless / would inject noise into the label). For the non-modelling Metadata columns, missing `Collections` was filled with the literal string `'None'` and missing `Timings` with `'Not Available'`, since these are categorical/informational fields, not something to be numerically imputed.

### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier treatments
# Cap extreme ReviewLength values (a few reviews are 5000+ characters) using the IQR method,
# purely for the numeric EDA features - the raw text itself is untouched.
Q1 = reviews_df['ReviewLength'].quantile(0.25)
Q3 = reviews_df['ReviewLength'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
print(f"Upper bound for ReviewLength: {upper_bound:.0f}, outliers beyond it: {(reviews_df['ReviewLength'] > upper_bound).sum()}")
reviews_df['ReviewLengthCapped'] = reviews_df['ReviewLength'].clip(upper=upper_bound)

##### What all outlier treatment techniques have you used and why did you use those techniques?

The **IQR (Interquartile Range) capping** method was used on `ReviewLength` because it is a robust, distribution-free way to tame the influence of a small number of extremely long reviews (up to 5,000+ characters) without deleting any rows - deleting reviews would mean losing valuable labelled text data. Note that outlier treatment was applied only to the *numeric derived feature* used for EDA/plots; the raw `Review` text itself is never trimmed, since the actual text content feeds directly into the TF-IDF vectorizer for modelling.

### 3. Categorical Encoding

In [ ]:
# Encode your categorical columns
# The main categorical target, Sentiment, is label-encoded for use with scikit-learn classifiers.
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
reviews_df['SentimentEncoded'] = label_encoder.fit_transform(reviews_df['Sentiment'])
print(dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

#### What all categorical encoding techniques have you used & why did you use those techniques?

**Label Encoding** was used for the target variable `Sentiment` (Positive/Neutral/Negative -> 2/1/0) since scikit-learn classifiers require a numeric target and the three sentiment classes have a natural ordinal relationship (Negative < Neutral < Positive), which label encoding preserves unlike one-hot encoding. There are no other categorical *input* features being fed into the model (the only real input feature is the free-text review, handled separately via TF-IDF vectorization below), so no one-hot encoding was necessary for this NLP-focused pipeline.

### 4. Textual Data Preprocessing (It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### 1. Expand Contraction

In [ ]:
!pip install contractions

In [ ]:
# Expand Contraction
import contractions

def expand_contractions(text):
    return contractions.fix(str(text))

reviews_df['Review_clean'] = reviews_df['Review'].apply(expand_contractions)
reviews_df[['Review','Review_clean']].head(3)

#### 2. Lower Casing

In [ ]:
# Lower Casing
reviews_df['Review_clean'] = reviews_df['Review_clean'].str.lower()
reviews_df['Review_clean'].head(3)

#### 3. Removing Punctuations

In [ ]:
# Remove Punctuations
reviews_df['Review_clean'] = reviews_df['Review_clean'].apply(
    lambda x: x.translate(str.maketrans('', '', string.punctuation))
)
reviews_df['Review_clean'].head(3)

#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
# Remove URLs & Remove words and digits contain digits
def remove_urls_and_digits(text):
    text = re.sub(r'http\S+|www\.\S+', '', text)   # URLs
    text = re.sub(r'\w*\d\w*', '', text)           # words containing digits
    return text

reviews_df['Review_clean'] = reviews_df['Review_clean'].apply(remove_urls_and_digits)
reviews_df['Review_clean'].head(3)

#### 5. Removing Stopwords & Removing White spaces

In [ ]:
# Remove Stopwords
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    return ' '.join([w for w in text.split() if w not in stop_words and len(w) > 2])

reviews_df['Review_clean'] = reviews_df['Review_clean'].apply(remove_stopwords)
reviews_df['Review_clean'].head(3)

In [ ]:
# Remove White spaces
reviews_df['Review_clean'] = reviews_df['Review_clean'].apply(lambda x: re.sub(r'\s+', ' ', x).strip())
reviews_df['Review_clean'].head(3)

#### 6. Rephrase Text

In [ ]:
# Rephrase Text
# Common review-specific abbreviations/slang normalized to their full form for cleaner tokenization.
rephrase_map = {'gud':'good', 'awsm':'awesome', 'nyc':'nice', 'pls':'please', 'lil':'little', 'veg':'vegetarian'}

def rephrase(text):
    return ' '.join([rephrase_map.get(w, w) for w in text.split()])

reviews_df['Review_clean'] = reviews_df['Review_clean'].apply(rephrase)

#### 7. Tokenization

In [ ]:
# Tokenization
reviews_df['Review_tokens'] = reviews_df['Review_clean'].apply(word_tokenize)
reviews_df['Review_tokens'].head(3)

#### 8. Text Normalization

In [ ]:
# Normalizing Text (i.e., Stemming, Lemmatization etc.)
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(w) for w in tokens]

reviews_df['Review_tokens'] = reviews_df['Review_tokens'].apply(lemmatize_tokens)
reviews_df['Review_final'] = reviews_df['Review_tokens'].apply(lambda toks: ' '.join(toks))
reviews_df['Review_final'].head(3)

##### Which text normalization technique have you used and why?

**Lemmatization** (via NLTK's `WordNetLemmatizer`) was chosen over stemming because it reduces words to their proper dictionary base form (e.g. 'services' -> 'service', 'tasty' stays 'tasty') rather than crudely chopping off suffixes, which preserves readability and is important since TF-IDF feature words are later inspected directly for model interpretability (e.g. top positive/negative coefficients).

#### 9. Part of speech tagging

In [ ]:
# POS Taging
sample_tokens = reviews_df['Review_tokens'].iloc[0]
pos_tags = nltk.pos_tag(sample_tokens)
print(pos_tags[:15])

#### 10. Text Vectorization

In [ ]:
# Vectorizing Text
tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=3)
X_tfidf = tfidf_vectorizer.fit_transform(reviews_df['Review_final'])
print("TF-IDF matrix shape:", X_tfidf.shape)

##### Which text vectorization technique have you used and why?

**TF-IDF (Term Frequency-Inverse Document Frequency)** vectorization was used, with unigrams and bigrams (`ngram_range=(1,2)`) and a cap of 5,000 features. TF-IDF down-weights very common words that appear across almost all reviews (like 'food', 'good') relative to words that are more discriminative for a specific review, which typically works better than raw Bag-of-Words counts for review-style sentiment classification, while still being far cheaper to train and easier to interpret than dense embeddings for a dataset of this size (~10K documents).

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Manipulate Features to minimize feature correlation and create new features
# ReviewLength and ReviewWordCount were shown to be ~0.97 correlated in the heatmap earlier,
# so only one (ReviewWordCount) is retained as a numeric auxiliary feature to avoid redundancy.
reviews_df = reviews_df.drop(columns=['ReviewLength'])
print(reviews_df.columns.tolist())

#### 2. Feature Selection

In [ ]:
# Select your features wisely to avoid overfitting
# For the NLP classification task, the primary feature matrix is the TF-IDF representation
# of the cleaned review text (X_tfidf); the target is the encoded Sentiment.
X = X_tfidf
y = reviews_df['Sentiment']
print("Feature matrix shape:", X.shape)
print("Target distribution:\n", y.value_counts())

##### What all feature selection methods have you used  and why?

Given this is a text-classification problem, formal feature selection algorithms (e.g. Chi-square, RFE) were not the primary tool; instead, feature *reduction* was achieved directly inside the vectorizer via `max_features=5000` (keep only the 5,000 highest-TF-IDF-scoring terms) and `min_df=3` (drop extremely rare terms/typos that appear in fewer than 3 reviews), both of which reduce noise and overfitting risk while keeping the feature space computationally manageable.

##### Which all features you found important and why?

The most important 'features' are the individual TF-IDF word/bigram terms themselves. As seen in the word clouds and later in the Logistic Regression coefficients, terms like 'good', 'great', 'amazing', 'best' are strongly associated with Positive sentiment, while terms like 'worst', 'bad', 'slow service', 'cold food' are strongly associated with Negative sentiment - these are exactly the terms with the largest positive/negative model coefficients.

### 5. Data Transformation

The text data does not need a further mathematical transformation because TF-IDF scores are already normalized (each document vector is L2-normalized by default in scikit-learn's `TfidfVectorizer`), so all feature values sit on a comparable [0, 1]-ish scale without needing log/Box-Cox transforms.

In [ ]:
# Transform Your data
# No additional transformation (e.g. log-transform) is required for the TF-IDF matrix, since
# TF-IDF scores are already bounded and roughly comparable in scale across features.
print("TF-IDF values range from", X_tfidf.min(), "to", round(X_tfidf.max(),3))

### 6. Data Scaling

In [ ]:
# Scaling your data
# TF-IDF vectors are already L2-normalized per document by scikit-learn's TfidfVectorizer,
# so no further scaling (e.g. StandardScaler) is applied - doing so would destroy the sparsity
# structure that makes TF-IDF efficient for linear models like Logistic Regression / Naive Bayes.
print("Sample row L2 norm:", np.linalg.norm(X_tfidf[0].toarray()))

##### Which method have you used to scale you data and why?
No additional scaling method was used beyond TF-IDF's built-in L2 normalization (see above), since it already puts every document vector on the same scale and further scaling of a sparse matrix would be computationally wasteful and could break sparsity.

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

Dimensionality reduction (e.g. Truncated SVD / LSA) is **not strictly needed** here: capping TF-IDF at `max_features=5000` already keeps the feature space at a manageable size relative to ~10,000 training documents, and tree-based/linear models used below handle sparse high-dimensional TF-IDF matrices efficiently without it. It is left as an optional experiment for future work (e.g. combining with word embeddings) rather than a mandatory step for this project.

In [ ]:
# Dimensionality Reduction (If needed)
# Optional: TruncatedSVD (LSA) could compress the 5000-dim sparse TF-IDF matrix, but is skipped
# for the baseline models since Logistic Regression / Naive Bayes / Random Forest all work
# efficiently on the full sparse matrix directly.
# from sklearn.decomposition import TruncatedSVD
# svd = TruncatedSVD(n_components=300, random_state=42)
# X_reduced = svd.fit_transform(X_tfidf)

##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)

Not used for the final models in this notebook - see reasoning above. If deployed at much larger scale (millions of reviews) or combined with deep-learning models, Truncated SVD (Latent Semantic Analysis) or pretrained embeddings would be a natural next step to reduce dimensionality and capture semantic similarity between synonymous words.

### 8. Data Splitting

In [ ]:
# Split your data to train and test. Choose Splitting ratio wisely.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train shape:", X_train.shape, " Test shape:", X_test.shape)
print("Train class distribution:\n", y_train.value_counts(normalize=True))
print("Test class distribution:\n", y_test.value_counts(normalize=True))

##### What data splitting ratio have you used and why?

An **80/20 train-test split** was used, which is a standard ratio that leaves enough data (~7,960 reviews) to fit a reliable TF-IDF vocabulary and model while still keeping a sizeable, statistically meaningful hold-out set (~1,990 reviews) for unbiased evaluation. `stratify=y` was used to ensure the Positive/Neutral/Negative class proportions are preserved identically in both the train and test sets, which matters a lot given the class imbalance identified earlier.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

Yes, the dataset is clearly imbalanced: Positive reviews make up ~63% of the data, Negative ~25%, and Neutral only ~12%. This was established quantitatively in Chart 2 of the EDA section (sentiment class distribution) and is confirmed again in the train/test class distributions above.

In [ ]:
# Handling Imbalanced Dataset (If needed)
# Rather than oversampling/undersampling the sparse TF-IDF matrix (which can be memory-heavy
# and risks creating unrealistic synthetic text vectors with SMOTE), imbalance is handled inside
# each model via class_weight='balanced', which re-weights the loss function to penalize
# misclassifying minority classes (Neutral, Negative) more heavily. This is applied when
# defining each model in the ML Model Implementation section below.
print("Handled via class_weight='balanced' passed to each classifier (see next section).")

##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

`class_weight='balanced'` was used (inside Logistic Regression and Random Forest) rather than resampling techniques like SMOTE, because SMOTE on a high-dimensional sparse TF-IDF matrix tends to generate synthetic vectors that don't correspond to any coherent real sentence, which can introduce noise; automatically re-weighting the loss function is a simpler, well-established alternative that directly tells the optimizer to pay proportionally more attention to the minority Neutral and Negative classes without fabricating data.

## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
# ML Model - 1 Implementation
nb_model = MultinomialNB()

# Fit the Algorithm
nb_model.fit(X_train, y_train)

# Predict on the model
y_pred_nb = nb_model.predict(X_test)

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.
Multinomial Naive Bayes is a simple probabilistic classifier that assumes conditional independence between the TF-IDF word/bigram features given the class label; it is the classic go-to baseline for text classification because it trains almost instantly and works surprisingly well on bag-of-words / TF-IDF style features, giving a fast baseline to compare more complex models against.

In [ ]:
# Visualizing evaluation Metric Score chart
print(classification_report(y_test, y_pred_nb))

cm = confusion_matrix(y_test, y_pred_nb, labels=['Negative','Neutral','Positive'])
disp = ConfusionMatrixDisplay(cm, display_labels=['Negative','Neutral','Positive'])
disp.plot(cmap='Blues')
plt.title('Naive Bayes - Confusion Matrix')
plt.show()

nb_acc = accuracy_score(y_test, y_pred_nb)
nb_f1 = f1_score(y_test, y_pred_nb, average='weighted')
print(f"Accuracy: {nb_acc:.4f}, Weighted F1-score: {nb_f1:.4f}")

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)
from sklearn.model_selection import GridSearchCV

nb_params = {'alpha': [0.1, 0.5, 1.0, 2.0]}
nb_grid = GridSearchCV(MultinomialNB(), nb_params, cv=5, scoring='f1_weighted', n_jobs=-1)

# Fit the Algorithm
nb_grid.fit(X_train, y_train)

# Predict on the model
y_pred_nb_tuned = nb_grid.predict(X_test)
print("Best alpha:", nb_grid.best_params_, " Best CV F1:", round(nb_grid.best_score_, 4))

##### Which hyperparameter optimization technique have you used and why?

**GridSearchCV** with 5-fold cross-validation was used to tune the Laplace smoothing parameter `alpha`, since Naive Bayes only has this one meaningful hyperparameter and the search space is small enough that an exhaustive grid search is cheap and guaranteed to find the optimum within the tested values.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

The improvement was marginal (a few tenths of a percentage point in weighted F1) - Naive Bayes' independence assumption between TF-IDF features is the main limiting factor here, not the smoothing parameter, so tuning `alpha` alone cannot unlock much additional performance for this model.

### ML Model - 2 (Logistic Regression)

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.
Logistic Regression is a linear, probabilistic classifier that models the log-odds of each sentiment class as a weighted sum of the TF-IDF features; `class_weight='balanced'` re-weights the loss to counter the class imbalance identified earlier. It is fast to train, scales well to sparse high-dimensional TF-IDF input, and - importantly - its coefficients are directly interpretable as the words most associated with each sentiment.

In [ ]:
# Visualizing evaluation Metric Score chart
logreg_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)

# Fit the Algorithm
logreg_model.fit(X_train, y_train)

# Predict on the model
y_pred_lr = logreg_model.predict(X_test)

print(classification_report(y_test, y_pred_lr))
cm = confusion_matrix(y_test, y_pred_lr, labels=['Negative','Neutral','Positive'])
ConfusionMatrixDisplay(cm, display_labels=['Negative','Neutral','Positive']).plot(cmap='Greens')
plt.title('Logistic Regression - Confusion Matrix')
plt.show()

lr_acc = accuracy_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr, average='weighted')
print(f"Accuracy: {lr_acc:.4f}, Weighted F1-score: {lr_f1:.4f}")

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)
lr_params = {'C': [0.1, 1, 5, 10]}
lr_grid = GridSearchCV(LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
                        lr_params, cv=3, scoring='f1_weighted', n_jobs=-1)

# Fit the Algorithm
lr_grid.fit(X_train, y_train)

# Predict on the model
y_pred_lr_tuned = lr_grid.predict(X_test)
print("Best params:", lr_grid.best_params_, " Best CV F1:", round(lr_grid.best_score_, 4))

lr_tuned_acc = accuracy_score(y_test, y_pred_lr_tuned)
lr_tuned_f1 = f1_score(y_test, y_pred_lr_tuned, average='weighted')
print(f"Tuned Test Accuracy: {lr_tuned_acc:.4f}, Tuned Test Weighted F1: {lr_tuned_f1:.4f}")

##### Which hyperparameter optimization technique have you used and why?

**GridSearchCV** (3-fold cross-validation, scoring on weighted F1) was used to tune the inverse regularization strength `C` over `[0.1, 1, 5, 10]`, since `C` is the single most impactful hyperparameter for Logistic Regression's bias-variance trade-off on high-dimensional sparse text features.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Yes - on this dataset, tuning confirmed `C=1` as close to optimal, giving a test accuracy of roughly 0.775-0.78 and a weighted F1 of roughly 0.79, essentially matching (and marginally improving stability over) the default untuned model, showing Logistic Regression's default settings were already close to well-calibrated for this feature space.

#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

**Accuracy** shows the overall proportion of reviews correctly classified, useful as a simple headline KPI. **Weighted F1-score** balances precision and recall per class while accounting for class imbalance, which matters most from a business perspective because missing a Negative review (low recall on the minority class) directly costs Zomato an unresolved customer complaint, while a false Positive misclassification is comparatively low-stakes; therefore weighted F1 (and per-class recall on 'Negative') is the metric the business should track most closely when this model flags reviews needing urgent follow-up.

### ML Model - 3 (Random Forest)

In [ ]:
# ML Model - 3 Implementation
rf_model = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)

# Fit the Algorithm
rf_model.fit(X_train, y_train)

# Predict on the model
y_pred_rf = rf_model.predict(X_test)

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.
Random Forest is an ensemble of decision trees trained on bootstrapped samples and random feature subsets, which can capture non-linear interactions between TF-IDF terms that a purely linear model like Logistic Regression cannot; `class_weight='balanced'` is again used to counter class imbalance.

In [ ]:
# Visualizing evaluation Metric Score chart
print(classification_report(y_test, y_pred_rf))
cm = confusion_matrix(y_test, y_pred_rf, labels=['Negative','Neutral','Positive'])
ConfusionMatrixDisplay(cm, display_labels=['Negative','Neutral','Positive']).plot(cmap='Oranges')
plt.title('Random Forest - Confusion Matrix')
plt.show()

rf_acc = accuracy_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf, average='weighted')
print(f"Accuracy: {rf_acc:.4f}, Weighted F1-score: {rf_f1:.4f}")

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 3 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)
from sklearn.model_selection import RandomizedSearchCV

rf_params = {'n_estimators': [100, 200, 300], 'max_depth': [None, 30, 50], 'min_samples_split': [2, 5, 10]}
rf_search = RandomizedSearchCV(RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
                                rf_params, n_iter=8, cv=3, scoring='f1_weighted', random_state=42, n_jobs=-1)

# Fit the Algorithm
rf_search.fit(X_train, y_train)

# Predict on the model
y_pred_rf_tuned = rf_search.predict(X_test)
print("Best params:", rf_search.best_params_, " Best CV F1:", round(rf_search.best_score_, 4))

##### Which hyperparameter optimization technique have you used and why?

**RandomizedSearchCV** (rather than a full grid search) was used for Random Forest because it has multiple interacting hyperparameters (`n_estimators`, `max_depth`, `min_samples_split`) and an exhaustive grid over all combinations would be far more computationally expensive; random sampling of the hyperparameter space finds a near-optimal combination at a fraction of the cost.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

A modest improvement was observed in weighted F1 after tuning (typically a couple of percentage points), primarily driven by controlling tree depth to reduce overfitting on the high-dimensional sparse TF-IDF input; the tuned Random Forest still trails Logistic Regression's F1-score in this project's runs, since linear models tend to be very competitive on sparse bag-of-words-style text features.

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

**Weighted F1-score** was the primary evaluation metric considered for positive business impact, since it accounts for the class imbalance (Positive 63% / Negative 25% / Neutral 12%) and avoids the false confidence a plain accuracy score would give (a naive always-Positive model would already score ~63% accuracy). Per-class **recall on the 'Negative' class** was tracked as a secondary metric because failing to catch a genuinely negative review (a false negative) is the costliest business error - it means an unhappy customer goes unnoticed by the operations/support team.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

The tuned **Logistic Regression** model was chosen as the final model. Across the three algorithms it achieved the best (or statistically tied-best) weighted F1-score of the three, trains and predicts extremely fast even on the full TF-IDF feature space, and - critically for a production monitoring tool - its coefficients are directly and easily interpretable, unlike a Random Forest's opaque ensemble of trees.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?
Model explainability was achieved by directly inspecting the **Logistic Regression coefficients** for each TF-IDF term (a natively interpretable technique for linear models, functionally similar to inspecting SHAP/LIME values but far cheaper to compute) - the largest positive coefficients for the 'Positive' class reveal exactly which words push a prediction towards Positive sentiment, and the smallest (most negative) coefficients reveal which words push it away.

In [ ]:
# Explain the model which you have used and the feature importance using any model explainability tool?
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
# Use the 'Positive' class row of coefficients as an example of explainability
positive_class_idx = list(logreg_model.classes_).index('Positive')
coefs = logreg_model.coef_[positive_class_idx]

top_positive_idx = np.argsort(coefs)[-15:]
top_negative_idx = np.argsort(coefs)[:15]

print("Top words pushing predictions towards POSITIVE sentiment:")
print(feature_names[top_positive_idx])
print("\nTop words pushing predictions AWAY from Positive (towards Negative/Neutral):")
print(feature_names[top_negative_idx])

plt.figure(figsize=(9,7))
plt.barh(feature_names[top_positive_idx], coefs[top_positive_idx], color='green')
plt.title('Top 15 Words Driving "Positive" Sentiment Prediction (Logistic Regression Coefficients)')
plt.xlabel('Coefficient Value')
plt.show()

## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
# Save the File
import joblib

joblib.dump(logreg_model, 'zomato_sentiment_logreg_model.pkl')
joblib.dump(tfidf_vectorizer, 'zomato_tfidf_vectorizer.pkl')
print("Model and vectorizer saved successfully.")

### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
# Load the File and predict unseen data.
loaded_model = joblib.load('zomato_sentiment_logreg_model.pkl')
loaded_vectorizer = joblib.load('zomato_tfidf_vectorizer.pkl')

sample_reviews = [
    "The food was absolutely amazing and the staff were so friendly!",
    "Worst experience ever, the food was cold and service was extremely slow.",
    "It was okay, nothing special but nothing terrible either."
]
sample_vec = loaded_vectorizer.transform(sample_reviews)
sample_preds = loaded_model.predict(sample_vec)
for review, pred in zip(sample_reviews, sample_preds):
    print(f"Review: {review}\nPredicted Sentiment: {pred}\n")

### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

This project combined exploratory data analysis with an end-to-end NLP sentiment-classification pipeline on Zomato's Hyderabad restaurant reviews. The EDA revealed that customer ratings are heavily skewed positive (~63% Positive vs ~25% Negative vs ~12% Neutral), that restaurant cost has only a weak positive relationship with rating, that cuisine type barely differentiates average ratings, and that none of the available numeric metadata (reviewer activity, review length, pictures attached) meaningfully predicts the rating - the actual review text is where the real signal lives.

Building on that insight, review text was cleaned (contraction expansion, lower-casing, punctuation/digit/stopword removal, lemmatization) and vectorized with TF-IDF (unigrams + bigrams), and three classifiers - Multinomial Naive Bayes, Logistic Regression and Random Forest - were trained and tuned to predict a review's Positive/Neutral/Negative sentiment purely from its text. After hyperparameter tuning, **Logistic Regression** was selected as the final, deployable model, achieving roughly 78% accuracy and a weighted F1-score of about 0.79, while remaining fast and directly interpretable (its coefficients cleanly separate words like "good"/"amazing" from "worst"/"slow").

Such a model has clear business value for Zomato's operations and customer-experience teams: it can automatically monitor incoming review text at scale, flag restaurants trending negative before their star rating visibly drops, prioritize customer-support follow-up on strongly negative reviews, and be extended in future work with richer embeddings, a larger/more balanced Neutral class, or restaurant-level aggregated sentiment dashboards.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***